# ¿Hace falta entrenar el encoder con JEPA?

Este notebook es un **smoke de desarrollo sobre validation con una sola seed**. Compara siete métodos sobre la misma familia $K_\rho=\rho C$ y las mismas ventanas. Su objetivo es comprobar el pipeline y la legibilidad del reporte antes de abrir el held-out; no produce la conclusión final.

La métrica principal es el MAE de la estimación espectral continua de $\rho$. La clasificación entre cinco candidatos queda como diagnóstico secundario.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display

from koopman_jepa.analysis import calibration_statistics
from koopman_jepa.decay_benchmark import BASELINE_METHODS, run_decay_baseline_development
from koopman_jepa.phase_data import PhaseWindowConfig, make_decay_phase_tensor_dataset_splits

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
CONFIG_PATH = ROOT / 'configs' / 'koopman_decay_baselines.yaml'
with CONFIG_PATH.open(encoding='utf-8') as handle:
    raw = yaml.safe_load(handle)

rhos = tuple(float(rho) for rho in raw['rho_values'])
development_seed = int(raw['seeds'][0])
invalid_error = float(raw['evaluation']['invalid_active_span_absolute_error'])
assert raw['status'] == 'frozen_before_implementation'
assert tuple(raw['methods']) == BASELINE_METHODS
assert raw['splits']['heldout_access'] == 'after_methods_tests_and_config_are_frozen'
display(Markdown(f'**Config:** `{CONFIG_PATH.name}` · **seed de desarrollo:** `{development_seed}` · **held-out:** cerrado'))

## Qué separa cada baseline

| Método | Pregunta que responde |
|---|---|
| Oracle de fase | ¿La estimación y la evaluación matemática funcionan? |
| DMD crudo | ¿La dinámica ya es lineal en las ventanas? |
| PCA-3 + DMD | ¿Basta una compresión lineal del tamaño del latent? |
| CNN aleatoria + DMD | ¿La arquitectura sin entrenamiento ya produce features suficientes? |
| CNN supervisada + DMD | ¿Qué logra el mismo encoder si recibe la fase explícitamente? |
| JEPA + predictor aprendido | Método bajo estudio. |
| JEPA + DMD post-hoc | ¿El encoder es mejor que el predictor aprendido? |

PCA, CNN aleatoria y CNN supervisada se ajustan una sola vez para esta seed. JEPA se entrena por separado para cada $\rho$, porque sus pares temporales cambian.

In [ ]:
emission = PhaseWindowConfig(**raw['emission'], repeats_per_transition=1)
splits = make_decay_phase_tensor_dataset_splits(
    emission,
    rhos,
    train_repeats_per_transition=raw['splits']['train_repeats_per_transition'],
    validation_repeats_per_transition=raw['splits']['validation_repeats_per_transition'],
    seed=development_seed,
)
assert splits.heldout is None and splits.heldout_seed is None
benchmark = run_decay_baseline_development(raw, splits, seed=development_seed)
rows = list(benchmark.rows)
assert len(rows) == len(BASELINE_METHODS) * len(rhos)
print(f'Corridas evaluadas: {len(rows)}; held-out materializado: {splits.heldout is not None}')

## Resultado de validation

Cada fila resume los cinco valores de $\rho$ para un método. `Rango 3` indica en cuántas condiciones existe el subespacio espectral requerido. Un rango inválido no se descarta: aporta el error predeclarado `1.0` al MAE espectral.

In [ ]:
summary = {}
for method in BASELINE_METHODS:
    subset = sorted((row for row in rows if row['method'] == method), key=lambda row: row['rho'])
    truth = np.array([row['rho'] for row in subset], dtype=float)
    spectral = np.array([np.nan if row['spectral_rho_estimate'] is None else row['spectral_rho_estimate'] for row in subset], dtype=float)
    action = np.array([np.nan if row['action_rho_estimate'] is None else row['action_rho_estimate'] for row in subset], dtype=float)
    spectral_stats = calibration_statistics(truth, spectral, invalid_absolute_error=invalid_error)
    action_stats = calibration_statistics(truth, action, invalid_absolute_error=invalid_error)
    summary[method] = {
        'spectral_mae': spectral_stats['mae'],
        'action_mae': action_stats['mae'],
        'slope': spectral_stats['slope'],
        'r_squared': spectral_stats['r_squared'],
        'rank3': sum(row['active_rank'] == raw['evaluation']['active_rank_required_for_spectrum'] for row in subset),
        'median_action_error': float(np.median([row['true_action_error'] for row in subset])),
        'median_h8_error': float(np.median([row['rollout_errors'][8] for row in subset])),
    }

lines = [
    '| Método | MAE espectral | MAE acción | Pendiente | $R^2$ | Rango 3 | Error acción | Rollout $h=8$ |',
    '|---|---:|---:|---:|---:|---:|---:|---:|',
]
for method in BASELINE_METHODS:
    item = summary[method]
    slope = '—' if item['slope'] is None else f"{item['slope']:.3f}"
    r_squared = '—' if item['r_squared'] is None else f"{item['r_squared']:.3f}"
    lines.append(f"| `{method}` | {item['spectral_mae']:.3f} | {item['action_mae']:.3f} | {slope} | {r_squared} | {item['rank3']}/5 | {item['median_action_error']:.3f} | {item['median_h8_error']:.3f} |")
display(Markdown('\n'.join(lines)))

In [ ]:
labels = {
    'phase_oracle_ols': 'Oracle fase',
    'raw_window_dmd': 'DMD crudo',
    'pca3_dmd': 'PCA-3 + DMD',
    'random_cnn3_dmd': 'CNN aleatoria',
    'supervised_phase_cnn3_dmd': 'CNN supervisada',
    'jepa_learned_predictor': 'JEPA',
    'jepa_posthoc_dmd': 'JEPA + DMD',
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot([0, 1], [0, 1], '--', color='black', linewidth=1, label='ideal')
for method in BASELINE_METHODS:
    subset = sorted((row for row in rows if row['method'] == method), key=lambda row: row['rho'])
    estimates = [row['spectral_rho_estimate'] for row in subset]
    axes[0].plot(rhos, estimates, marker='o', label=labels[method])
axes[0].set(xlabel=r'$\rho$ verdadero', ylabel=r'$\widehat{\rho}$ espectral', title='Calibración espectral en validation')
axes[0].legend(fontsize=8, ncol=2)

maes = [summary[method]['spectral_mae'] for method in BASELINE_METHODS]
axes[1].barh([labels[method] for method in BASELINE_METHODS], maes, color=plt.cm.viridis(np.linspace(0.15, 0.9, len(maes))))
axes[1].invert_yaxis()
axes[1].set(xlabel='MAE espectral (menor es mejor)', title='Error agregado sobre cinco dinámicas')
fig.tight_layout()
plt.show()

In [ ]:
jepa_mae = summary['jepa_learned_predictor']['spectral_mae']
random_mae = summary['random_cnn3_dmd']['spectral_mae']
raw_mae = summary['raw_window_dmd']['spectral_mae']
posthoc_mae = summary['jepa_posthoc_dmd']['spectral_mae']
supervised_mae = summary['supervised_phase_cnn3_dmd']['spectral_mae']
best_unsupervised = min(('raw_window_dmd', 'pca3_dmd', 'random_cnn3_dmd'), key=lambda method: summary[method]['spectral_mae'])

if jepa_mae < random_mae:
    causal_read = f"En esta seed, JEPA mejora a la CNN aleatoria en {random_mae - jepa_mae:.3f} puntos de MAE."
else:
    causal_read = f"En esta seed, la CNN aleatoria iguala o mejora a JEPA por {jepa_mae - random_mae:.3f} puntos de MAE; todavía no hay atribución a JEPA."
predictor_read = (
    f"El DMD post-hoc mejora al predictor JEPA por {jepa_mae - posthoc_mae:.3f} puntos, lo que señala margen en la optimización de M."
    if posthoc_mae < jepa_mae
    else f"El predictor aprendido no queda por debajo del DMD post-hoc en esta seed (diferencia {posthoc_mae - jepa_mae:.3f})."
)
display(Markdown(
    '## Lectura provisional\n\n'
    + causal_read + '\n\n'
    + f"El mejor baseline no supervisado aquí es `{best_unsupervised}`. DMD crudo obtiene MAE {raw_mae:.3f}; el techo supervisado, {supervised_mae:.3f}.\n\n"
    + predictor_read + '\n\n'
    + '**Esto no decide la hipótesis:** es una sola seed de validation. La comparación causal predeclarada requiere las diez diferencias pareadas del held-out.'
))

## Próximo paso

Si el oracle y el reporte son coherentes, se congela este pipeline. Después se materializa el held-out una sola vez para las diez seeds y se calcula el contraste pareado `CNN aleatoria − JEPA`. No se retocan hiperparámetros a partir de este notebook.